# ДЗ 14. NumPy: векторизация, broadcasting, оси, маски 🔢
Решите задачи в Google Colab и сдайте через GitHub: папка `hw14` в репозитории `cyberformula3-homework`.

- **[База]** — обязательно для всех ·
- **[Челлендж]** — сколько получится ·
- **[Про]** — для тех, кому легко ·

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы и маски. Цикл можно там, где это сказано.

**Проверь себя перед сдачей:**
□ **Среда выполнения → Перезапустить и выполнить все** — ноутбук проходит без ошибок
□ Ответы совпадают с «Примером проверки», где он есть

In [1]:
import numpy as np
# проверка: вставьте сюда строки 1–5 и запустите

## Загрузка данных — запустите как есть

In [2]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders.csv"
print("Работаем с файлом:", FILE)

Saving orders.csv to orders.csv
Работаем с файлом: orders.csv


In [3]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [13]:
# TODO
new_revenue = revenue * 1.05
growth = new_revenue.sum() - revenue.sum()
print(f"Выручка вырастет на {growth:,.0f} ₸".replace(",", " "))

Выручка вырастет на 4 625 325 ₸


### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [14]:
# TODO
smart = category == "Смартфоны"
n_smart = smart.sum()
smart_revenue = revenue[smart].sum()
smart_share = smart.mean()

print("Заказов:", n_smart)
print(f"Сумма: {smart_revenue:,.0f} ₸".replace(",", " "))
print(f"Доля: {smart_share:.1%}")

Заказов: 131
Сумма: 30 689 000 ₸
Доля: 26.2%


### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [15]:
# TODO
mask = (status == "оплачен") & (city == "Шымкент")

print("Заказов:", mask.sum())
print(f"Сумма: {revenue[mask].sum():,.0f} ₸".replace(",", " "))

Заказов: 71
Сумма: 12 852 500 ₸


### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [16]:
w = revenue[:30].reshape(6, 5)
# TODO
day_sums = w.sum(axis=1)
print("Суммы по дням:", day_sums, "- чисел:", len(day_sums))

best_day = day_sums.argmax() + 1
print(f"Лучший день: {best_day}-й, {day_sums.max():,.0f} ₸".replace(",", " "))

print("Самые дорогие по дням:", w.max(axis=1))

Суммы по дням: [ 699000  925000 1094000  636000 1047000  571000] - чисел: 6
Лучший день: 3-й  1 094 000 ₸
Самые дорогие по дням: [198000 319000 549000 449000 449000 319000]


## [Челлендж]

### Задача 5. Заказы в трёх валютах
Покупатели из-за рубежа просят показать суммы заказов в своей валюте.
Курсы Нацбанка РК на 19 ноября 2026: 1 USD = 444.16 ₸, 1 EUR = 487.29 ₸, 1 RUB = 5.40 ₸.

1. Возьмите суммы **первых 5 заказов** из `revenue` и получите **одной операцией** таблицу «заказ × валюта» формы `(5, 3)`, округлите до 2 знаков.
   Какую форму нужно дать суммам заказов, чтобы сработал broadcasting?
2. Выведите рядом товар каждого из этих 5 заказов — массив `product` уже собран в ячейке загрузки.


**Пример проверки:** форма суммам — `(5, 1)`; первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — `[445.79  406.33  36666.67]`.

In [8]:
# TODO

### Задача 6. Категории
Для **каждой** категории из `np.unique(category)` выведите число заказов, выручку и средний чек (цикл по категориям можно).
Какая категория приносит больше всего денег? А у какой самый большой средний чек?

**Пример проверки:** и по выручке, и по среднему чеку лидируют **Ноутбуки** — `40 992 000 ₸`, средний чек ≈ `422 598 ₸`.

In [9]:
# TODO

### Задача 7. Бесплатная доставка
Доставка бесплатна, если сумма заказа от 50 000 ₸. С помощью `np.where` подпишите каждый заказ: `"бесплатно"` или `"платно"`.
Сколько заказов получили бесплатную доставку?

**Пример проверки:** `354`.

In [10]:
# TODO

## [Про]

### Задача 8. Таблица «категория × месяц»
Постройте таблицу **числа заказов** формы `(5, 8)`: строки — категории (`np.unique(category)`), столбцы — месяцы (`np.unique(month)`).
Двойной цикл по категориям и месяцам можно, внутри — маска из двух условий.

1. Проверьте: сумма всей таблицы должна быть равна числу заказов.
2. В каком месяце было больше всего заказов смартфонов?

**Пример проверки:** сумма таблицы `500`; больше всего смартфонов — в **сентябре (9)**, `22` заказа.

In [11]:
# TODO

### Задача 9. Доли категорий по месяцам
Разделите таблицу на итог по **каждому месяцу** — получится доля каждой категории внутри месяца.
Это broadcasting: таблица `(5, 8)` делится на строку `(8,)`. Какая `axis` даёт итог по месяцам?

1. Проверьте: сумма каждого столбца после деления равна `1`.
2. Какая доля заказов в **августе** пришлась на аксессуары?
3. Сохраните доли в `shares.json`: `{"категории": [...], "месяцы": [...], "доли": [[...], ...]}`, доли округлите до 2 знаков. Не забудьте `.tolist()`.

**Пример проверки:** в августе аксессуары — `40.4%` заказов.

In [12]:
# TODO